# Q1. Supervised Learning — Heart Disease Classification

**Goal:** Build and evaluate classification models to predict whether a patient has heart disease.  
**Target column:** `heart_disease` (1 = disease present, 0 = absent)  
**Dataset:** `q1_heart_disease.csv` — 800 rows, 12 columns

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')
print('All libraries loaded successfully.')

---
## Task 1 — Data Loading and Inspection

We load the dataset and inspect its shape, data types, and missing value counts to understand what preprocessing steps are needed.

In [ ]:
df = pd.read_csv('../data/q1_heart_disease.csv')

print('=' * 55)
print(f'  Dataset Shape: {df.shape[0]} rows  x  {df.shape[1]} columns')
print('=' * 55)

print('\n--- Data Types ---')
print(df.dtypes)

print('\n--- Missing Value Counts ---')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

print('\n--- First 5 Rows ---')
df.head()

---
## Task 2 — Exploratory Data Analysis

We produce three visualisations to understand the data before modelling:
1. **Target class distribution** — check for class imbalance
2. **Correlation heatmap** — identify which numerical features relate to the target
3. **Age distribution by heart disease status** — assess discriminative power of age

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: Target Class Distribution
counts = df['heart_disease'].value_counts().sort_index()
axes[0].bar(['No Disease (0)', 'Disease (1)'], counts.values,
            color=['steelblue', 'tomato'], edgecolor='black', width=0.5)
axes[0].set_title('Target Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Patients')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold', fontsize=11)

# ── Plot 2: Correlation Heatmap (numerical features only)
num_cols_plot = df.select_dtypes(include='number').columns
corr = df[num_cols_plot].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=axes[1], square=True, linewidths=0.5, annot_kws={'size': 8})
axes[1].set_title('Correlation Heatmap\n(Numerical Features)', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

# ── Plot 3: Age Distribution by Target Class
df[df['heart_disease'] == 0]['age'].plot(
    kind='hist', bins=20, alpha=0.6, color='steelblue', label='No Disease', ax=axes[2])
df[df['heart_disease'] == 1]['age'].plot(
    kind='hist', bins=20, alpha=0.6, color='tomato', label='Disease', ax=axes[2])
axes[2].set_title('Age Distribution by\nHeart Disease Status', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.tight_layout()
plt.show()

### EDA Interpretation

**Plot 1 — Target Class Distribution:**  
The dataset is nearly balanced — 407 patients with heart disease (51%) vs 393 without (49%). This is ideal for classification: no class imbalance correction (e.g. SMOTE or class weighting) is required, and accuracy remains a meaningful metric alongside F1-score.

**Plot 2 — Correlation Heatmap:**  
Among the numerical features, `oldpeak` (ST depression) shows the strongest positive correlation with `heart_disease`, meaning higher ST depression values are associated with disease. `max_hr` has a notable negative correlation — patients with lower maximum heart rates are more likely to have disease. `age` shows a mild positive correlation. `resting_bp` and `cholesterol` show weak individual correlations, suggesting they contribute only in combination with other features.

**Plot 3 — Age Distribution by Heart Disease Status:**  
Patients with heart disease tend to be slightly older on average. However, the distributions overlap substantially across all age groups, confirming that age alone is insufficient for classification. It carries useful signal, but the model must leverage multiple features in combination to achieve strong predictive performance.

---
## Task 3 — Data Preprocessing

Preprocessing follows four steps in order:
1. Handle missing values
2. One-hot encode categorical variables
3. Scale numerical features
4. Stratified train/test split

### Missing Value Strategy — Median Imputation

Two columns contain missing values: `resting_bp` (24 missing) and `cholesterol` (32 missing). Both are continuous numerical medical measurements.

**Why median imputation?**
- Medical measurements can be skewed by physiological outliers (e.g., extreme cholesterol in patients with disorders). The median is robust to such outliers, unlike the mean.
- Dropping rows would discard ~7% of data, unnecessarily reducing the training set.
- The missingness appears to be random (not systematic), so imputation is appropriate.

In [ ]:
df_clean = df.copy()

# Modern pandas-safe imputation (avoids copy-on-write ChainedAssignmentError)
df_clean['resting_bp']  = df_clean['resting_bp'].fillna(df_clean['resting_bp'].median())
df_clean['cholesterol'] = df_clean['cholesterol'].fillna(df_clean['cholesterol'].median())

print('Missing values after imputation:')
print(df_clean.isnull().sum())
print(f'\nTotal missing: {df_clean.isnull().sum().sum()} — all resolved.')

In [ ]:
# Define column groups
cat_cols = ['chest_pain_type', 'resting_ecg', 'st_slope']
num_cols = ['age', 'sex', 'resting_bp', 'cholesterol', 'fasting_bs',
            'max_hr', 'exercise_angina', 'oldpeak']

# One-hot encode categorical features (drop_first=False to retain all categories)
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=False)
print(f'Columns after one-hot encoding: {df_encoded.shape[1]}')
print('New columns:', [c for c in df_encoded.columns if c not in df_clean.columns])

# Separate features and target
X = df_encoded.drop(columns=['heart_disease'])
y = df_encoded['heart_disease']

# Stratified 80/20 split — stratify=y preserves class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTraining set : {X_train.shape[0]} rows')
print(f'Test set     : {X_test.shape[0]} rows')
print(f'\nTrain class distribution:\n{y_train.value_counts()}')
print(f'Test class distribution:\n{y_test.value_counts()}')

# Scale numerical features — fit on train, apply to both (prevents data leakage)
scaler = StandardScaler()
X_train = X_train.copy()
X_test  = X_test.copy()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

print(f'\nNaN in X_train: {X_train.isnull().sum().sum()}')
print(f'NaN in X_test : {X_test.isnull().sum().sum()}')
print('\nPreprocessing complete. Data is ready for modelling.')

---
## Task 4 — Model Training

We train three scikit-learn classifiers, all with `random_state=42` for reproducibility:

| Model | Key Idea |
|---|---|
| **Decision Tree** | Single tree that splits on feature thresholds. Simple, interpretable, prone to overfitting. |
| **Random Forest** | Ensemble of 100 trees trained on random subsets (bagging). Reduces variance. |
| **Gradient Boosting** | Sequential ensemble where each tree corrects the errors of the previous. Strong at reducing bias. |

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42)
gb = GradientBoostingClassifier(random_state=42)

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

print('All three models trained successfully.')

---
## Task 5 — Model Evaluation

For each model we compute the confusion matrix and print Precision, Recall, and F1-score.

**Why these metrics (not just accuracy)?**  
In a medical diagnosis context, false negatives (missing a disease case) and false positives (incorrectly flagging a healthy patient) carry different clinical costs. Precision, Recall, and F1-score expose these trade-offs explicitly — accuracy alone can be misleading, particularly if class imbalance exists.

In [ ]:
models = {
    'Decision Tree':     dt,
    'Random Forest':     rf,
    'Gradient Boosting': gb
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
results = {}

for i, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test)

    p  = precision_score(y_test, y_pred)
    r  = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results[name] = {'Precision': p, 'Recall': r, 'F1-Score': f1}

    # Confusion matrix heatmap
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Pred: No Disease', 'Pred: Disease'],
                yticklabels=['True: No Disease', 'True: Disease'])
    axes[i].set_title(f'{name}\nP={p:.3f}  R={r:.3f}  F1={f1:.3f}',
                      fontweight='bold', fontsize=10)

    print(f'\n{"="*45}')
    print(f'  Model: {name}')
    print(f'  Precision : {p:.4f}')
    print(f'  Recall    : {r:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

plt.suptitle('Confusion Matrices — All Models', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n--- Summary Table ---')
results_df = pd.DataFrame(results).T.round(4)
print(results_df)

### Model Evaluation — Interpretation

Based on the metrics above:

| Model | Precision | Recall | F1-Score |
|---|---|---|---|
| Decision Tree | 0.7195 | 0.7284 | 0.7239 |
| Random Forest | 0.7765 | 0.8148 | 0.7952 |
| Gradient Boosting | 0.7778 | 0.7778 | 0.7778 |

**Best model: Random Forest** — it achieves the highest F1-score (0.7952) and the highest Recall (0.8148).  

In a medical context, **Recall** is particularly important — it measures how many actual disease cases are correctly identified (true positive rate). A high Recall means fewer patients with real heart disease are missed. Random Forest's bagging approach reduces overfitting compared to the single Decision Tree, and in this dataset it outperforms Gradient Boosting's sequential approach as well.  

The Decision Tree is the weakest performer — without ensemble averaging, it overfits the training data and generalises poorly to unseen patients. **Random Forest is selected for hyperparameter tuning in Task 6.**

---
## Task 6 — Hyperparameter Tuning (Random Forest)

`GridSearchCV` with 5-fold cross-validation systematically searches for the best combination of:
- `n_estimators` — number of trees in the forest
- `max_depth` — maximum depth of each tree (controls overfitting)
- `min_samples_split` — minimum samples required to split a node

Scoring is `f1` to align with our primary evaluation metric.

In [ ]:
# Baseline performance of the untuned Random Forest
y_pred_baseline = rf.predict(X_test)
f1_baseline = f1_score(y_test, y_pred_baseline)
print(f'Baseline Random Forest F1-Score (untuned): {f1_baseline:.4f}')

# Hyperparameter search space
param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [None, 5, 10],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'\nBest Parameters Found: {grid_search.best_params_}')
print(f'Best CV F1-Score      : {grid_search.best_score_:.4f}')

In [ ]:
# Evaluate tuned model on held-out test set
best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)
f1_tuned = f1_score(y_test, y_pred_tuned)

print('\n--- Tuned vs Baseline Comparison ---')
print(f'Baseline RF F1-Score : {f1_baseline:.4f}')
print(f'Tuned RF F1-Score    : {f1_tuned:.4f}')
print(f'Improvement          : {f1_tuned - f1_baseline:+.4f}')

print('\nFull Classification Report — Tuned Random Forest:')
print(classification_report(y_test, y_pred_tuned, target_names=['No Disease', 'Disease']))

### Hyperparameter Tuning — Interpretation

`GridSearchCV` evaluated 18 parameter combinations (3 × 3 × 2) across 5 folds (90 fits total), scoring each by F1-score.

Controlling `max_depth` and `min_samples_split` prevents individual trees in the forest from growing too deep and overfitting the training set. Increasing `n_estimators` generally reduces variance up to a point.  

The tuned model's F1-score on the test set is compared against the untuned baseline. Even a modest positive delta confirms that scikit-learn's default hyperparameters are not optimal and that systematic grid search improves generalisation to unseen patient data.  

The tuned Random Forest is our **final recommended model** for heart disease classification.